In [ ]:
# 1. Import libraries
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import time


In [ ]:
# ==================================================
# 2. LOAD DATASET
# ==================================================

train = pd.read_csv("dataset/Handwritten/mnist_train.csv")
test = pd.read_csv("dataset/Handwritten/mnist_test.csv")

In [ ]:
# ==================================================
# 3. DATA PREPROCESSING
# ==================================================

# 3.1 Check missing values
print("\nMissing values in training data:", train.isnull().sum().sum())
print("Missing values in testing data:", test.isnull().sum().sum())


# 3.2 Handle missing values
if train.isnull().sum().sum() > 0:
    train = train.fillna(train.median(numeric_only=True))

if test.isnull().sum().sum() > 0:
    test = test.fillna(test.median(numeric_only=True))


# 3.3 Check duplicate rows
print("\nDuplicate rows in training data:", train.duplicated().sum())
print("Duplicate rows in testing data:", test.duplicated().sum())


# Remove duplicate rows
train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)


# 3.4 Display dataset information
print("\nTraining dataset information:")
print(train.info())


# 3.5 Separate features and labels
x_train = train.iloc[:, 1:].values
y_train = train.iloc[:, 0].values

x_test = test.iloc[:, 1:].values
y_test = test.iloc[:, 0].values


# 3.6 Convert data type
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')


# 3.7 Normalize pixel values
x_train = x_train / 255.0
x_test = x_test / 255.0


# 3.8 Display dataset shapes
print("\nTraining data:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing data:", x_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
# ==================================================
# 4. RESHAPE DATA FOR CNN
# ==================================================

x_train_cnn = x_train.reshape(-1, 28, 28, 1)
x_test_cnn = x_test.reshape(-1, 28, 28, 1)

print("\nCNN Training image shape:", x_train_cnn.shape)
print("CNN Testing image shape:", x_test_cnn.shape)


In [ ]:
# ==================================================
# 5. BUILD AND TRAIN MLP MODEL
# ==================================================

mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

mlp.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start = time.time()

h1 = mlp.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

mlp_time = time.time() - start

mlp_loss, mlp_acc = mlp.evaluate(x_test, y_test, verbose=0)

print("\nMLP Accuracy:", round(mlp_acc * 100, 2), "%")
print("MLP Training Time:", round(mlp_time, 2), "seconds")

In [ ]:
# ==================================================
# 6. BUILD AND TRAIN CNN WITH DATA AUGMENTATION
# ==================================================

cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),

    # Data Augmentation
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomTranslation(0.1, 0.1),

    # First Convolution Layer
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Second Convolution Layer
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Fully Connected Layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start = time.time()

h2 = cnn.fit(
    x_train_cnn,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

cnn_time = time.time() - start

cnn_loss, cnn_acc = cnn.evaluate(x_test_cnn, y_test, verbose=0)

print("\nCNN Accuracy:", round(cnn_acc * 100, 2), "%")
print("CNN Training Time:", round(cnn_time, 2), "seconds")

In [ ]:
# ==================================================
# 7. COMPARE MLP AND CNN ACCURACY
# ==================================================

print("\nFinal Model Comparison")

print("MLP Accuracy:", round(mlp_acc * 100, 2), "%")
print("CNN Accuracy:", round(cnn_acc * 100, 2), "%")


plt.figure(figsize=(8, 5))

plt.bar(['MLP', 'CNN'], [mlp_acc * 100, cnn_acc * 100])

plt.title("MLP vs CNN Accuracy")
plt.xlabel("Models")
plt.ylabel("Accuracy (%)")
plt.grid(axis='y')
plt.show()

In [ ]:
# ==================================================
# 8. COMPARE TRAINING TIME
# ==================================================

plt.figure(figsize=(8, 5))

plt.bar(['MLP', 'CNN'], [mlp_time, cnn_time])

plt.title("Training Time Comparison")
plt.xlabel("Models")
plt.ylabel("Time (seconds)")
plt.grid(axis='y')
plt.show()


In [ ]:
# ==================================================
# 9. TRAINING ACCURACY COMPARISON
# ==================================================

plt.figure(figsize=(8, 5))

plt.plot(h1.history['accuracy'], label='MLP')
plt.plot(h2.history['accuracy'], label='CNN')

plt.title("Training Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ==================================================
# 10. VALIDATION ACCURACY COMPARISON
# ==================================================

plt.figure(figsize=(8, 5))

plt.plot(h1.history['val_accuracy'], label='MLP')
plt.plot(h2.history['val_accuracy'], label='CNN')

plt.title("Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ==================================================
# 11. VALIDATION LOSS COMPARISON
# ==================================================

plt.figure(figsize=(8, 5))

plt.plot(h1.history['val_loss'], label='MLP')
plt.plot(h2.history['val_loss'], label='CNN')

plt.title("Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()